# 04 - Practical MLOps in AzureML
---

### **Introduction**
Azure Machine Learning (AML) is a powerful tool designed to help facilitate the development of machine learning solutions in accordance with MLOps best-practices. It contains features and tools designed to help with reproducibility and monitoring. This notebook aims to be a practical guide for how to conduct the entire ML model lifecycle in AML. Note that the setup for AML including resource groups, workspaces, role management etc. is beyond the scope of this guide. Instead it will focus on practical ML delivery using AML. 

### **Interacting with a Workspace**
There are 3 ways to interact with an AML Workspace:
- **Studio**: The web-based application version where you use the GUI to perform operations. This is ideal for quick experimentation and development however for production level deployments or repetitive tasks you should use another option.
- **Azure CLI**: A set of command line operations for interacting with AML;refer to the [installation guide](https://learn.microsoft.com/en-us/cli/azure/install-azure-cli?view=azure-cli-latest#install). After this you will need to install the AML extension (`az extension add -n ml -y`). Refer to the documentation for specific commands. 
- **Python SDK**: A python package (`azure-ai-ml`) for interacting with AML from python. Refer to the [documentation](https://learn.microsoft.com/en-us/python/api/azure-ai-ml/azure.ai.ml.mlclient?view=azure-python) for specifics. 


As much as possible, we want to try to use the Python SDK to perform actions because:
- We can version control our infrastructure using Github so that we can roll back to a previous version if there is a mistake
- Recreating infrastructure is as simple as running a script, no complicated written instructions to try and follow 
- Provisioning large volumes of infrastructure can be automated (e.g. if we wanted to run many similar scripts for processing data)
- Data processing steps can be scheduled (e.g. overnight) so data is automatically updated when we need it

It is not absolutely necessary to use the Python SDK over the bash CLI; indeed, all the same functionality is available via the CLI. However generally developers find working in Python easier than bash so we think it makes sense to favour the Python SDK. 

To interact with AML via the Python SDK you need to create a client using your subscription ID, resource group name and AML workspace name.

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient(
    DefaultAzureCredential(),
    subscription_id="<subscription-id>",
    resource_group_name="<resource-group-name>",
    workspace_name="<workspace-name>",
)

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


### **Python SDK Authentication**
Typically when using the Python SDK we authenticate using the `DefaultAzureCredential` which works by attempting multiple different authentication methods:

1. **EnvironmentCredential**: Authenticates using a service principal define via environment variables:
    - AZURE_CLIENT_ID
    - AZURE_TENANT_ID
    - AZURE_CLIENT_SECRET 
2. **WorkloadIdentityCredential**: Uses federated identity for Kubernetes and CI/CD environments which works as follows:
    - GitHub runner requests an OIDC token from GitHub
    - GitHub signs it
    - Azure AD has a federated credential configured that trusts tokens from the given repo
    - Azure exchanges the GitHub OIDC token for an Azure access token
    - SDK uses that token to access AML
3. **ManagedIdentityCredential**: Used when running on an Azure resource that has a system-assigned or user-assigned managed identity (e.g., AML compute, VM, AKS). The SDK retrieves tokens via the Instance Metadata Service (IMDS).
4. **Developer Credentials** These may include:
    - AzureCliCredential → Uses identity from `az login`
    - VisualStudioCredential
    - VisualStudioCodeCredential
    - Azure Developer CLI (azd) credential

The auth method will typically differ depending on the situation:
- **Local Development**: Use the developer credentials
- **Testing a Docker container locally**: Pass environment variables from a service principal or to be more secure use a workload identity
- **Production Deployment**: Usually a managed identity

### **Data Storage**
AML provides the following means of accessing data for processing, training and inference:
- **Datastores**: Datastores securely connect to a storage service on Azure (e.g. Blob storage, Azure Data Lake Storage Gen2 (ADLS)) by storing connection information rather than storing the data directly
- **Data Assets**: Data assets are immutable references to your data that can be created from datastores, local files, public URLs, or Open Datasets
- **Data Import**: Data import creates data assets by importing from external storage services. You can create data assets from database services and filesystem services from other vendors outside of Azure
- **Data Connect**: Data connections securely connect to an external storage service by storing connection information. Data connections are needed to import data from storages external to Azure.

As default, each workspace comes with 4 datastores:
- **workspaceartifactstore**: Connects to the AML container of the Azure Storage account created with the workspace. Used to store compute and experiment logs when running jobs
- **workspaceworkingdirectory**: Connects to the file share of the Azure Storage account created with the workspace used by the Notebooks section of the studio. Whenever you upload files or folders to access from a compute instance, it's uploaded to this file share.
- **workspaceblobstore**: Connects to the Blob Storage of the Azure Storage account created with the workspace. Specifically the azureml-blobstore-... container. Set as the default datastore, which means that whenever you create a data asset and upload data, it's stored in this container.
- **workspacefilestore**: Connects to the file share of the Azure Storage account created with the workspace. Specifically the azureml-filestore-... file share.

Additionally, you can create datastores to connect to other Azure data services such as ADLS.

Note that OneLake datastores used to connect to data in your Microsoft Fabric endpoint can be created via CLI and SDK only. An example of how this can be done is shown below. 

```python
from azure.ai.ml import MLClient
from azure.ai.ml.entities import OneLakeDatastore, OneLakeArtifact
from azure.identity import DefaultAzureCredential

# Azure ML workspace details
SUBSCRIPTION_ID = "<azure-subscription-id>"
RESOURCE_GROUP = "<azure-ml-resource-group>"
WORKSPACE_NAME = "<azure-ml-workspace-name>"

# Fabric / OneLake details
DATASTORE_NAME = "fabric_lakehouse_ds"
ONE_LAKE_ENDPOINT = "onelake.dfs.fabric.microsoft.com"  # or your tenant-specific endpoint from Fabric Properties
ONE_LAKE_WORKSPACE_GUID = "<fabric-workspace-guid>"
LAKEHOUSE_ARTIFACT_GUID = "<lakehouse-guid>"

# Optional: point to a subfolder inside Files instead of the whole Files area
# Example: f"{LAKEHOUSE_ARTIFACT_GUID}/Files/myfolder"
ARTIFACT_PATH = f"{LAKEHOUSE_ARTIFACT_GUID}/Files"

credential = DefaultAzureCredential()

ml_client = MLClient(
    credential=credential,
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)

store = OneLakeDatastore(
    name=DATASTORE_NAME,
    description="Datastore pointing to a Microsoft Fabric lakehouse in OneLake",
    one_lake_workspace_name=ONE_LAKE_WORKSPACE_GUID,
    endpoint=ONE_LAKE_ENDPOINT,
    artifact=OneLakeArtifact(
        name=ARTIFACT_PATH,
        type="lake_house",
    ),
)

created = ml_client.datastores.create_or_update(store)

print("Created datastore:")
print(f"  Name: {created.name}")
print(f"  Type: {created.type}")
print(f"  ID:   {created.id}")
```

### **Environments**
Code used to process data, train models or run inference can be run as a command job in AML which is a containerised step. In order to run code as a command job, Azure needs all the elements required to execute your code including the operating system, supplementary code (e.g. specific utils functions you have written) as well as any dependencies. You can package all of this together using Docker into a docker image which is uploaded to the Azure Container Registry (ACR). An environment can the be defined by pointing it to this custom image. AML provides many curated environments which package up common groups of dependencies, however in general it is advised to build your own custom environments as it makes them more lightweight. Yoy may also want to define separate environments for data processing, model training and model inference for similar reasons. 

The steps to setting up separate custom environment are as follows:
1. Define a `pyproject.toml` file for your project which splits out dependencies into groups based on whether they will be used for data processing engineering, model training or model inference
2. Define a `Dockerfile` for each group of dependencies which installs the dependencies for that specific function (i.e. processing Dockerfile installs processing dependencies etc.)
3. Build your Docker images, naming and tagging them accordingly and push them to the ACR
4. Define environments using the python SDK, pointing them to the revant images in ACR

Note that you could simplify the above approach by combining the dependencies and using a single shared environment for each of the different ojb types (processing/training/inference). 

The example below demonstrates the approach for building 3 separate environments (feature_engineering, training, inference). It uses a `pyproject.toml` file with separate dependency groups for the different environments. The `Dockerfile` installs the relevant dependencies based on a `--build-arg`. The separate images are named and tagged accordingly before being pushed to the ACR. Finally the Python SDK is used to define 3 separate environments which will then be available to run the different command jobs for feature engineering, model training and model inference. 

`pyproject.toml`
```toml
[project]
name = "my_ml_project"
version = "0.1.0"
requires-python = ">=3.10"
dependencies = [
    "pandas>=2.3.2",
]

[project.optional-dependencies]
feature_engineering = [
    "numpy>=2.3.1",
]

training = [
    "scikit-learn>=1.7.1",
]

inference = [
    "fastapi>=0.115.8",
    "scikit-learn>=1.7.1",
    
]

[dependency-groups]
dev = [
    "pytest>=8.3.4",
    "pre-commit>=4.1.0",
    "nbstripout>=0.8.1",
]

[tool.setuptools.packages.find]
where = ["src/"]

[build-system]
requires = ["setuptools", "wheel"]
build-backend = "setuptools.build_meta"
```

`Dockerfile`

```docker
FROM python:3.10-slim-bookworm

# Set the working directory
WORKDIR /app

# The installer requires curl (and certificates) to download the release archive
RUN apt-get update && apt-get install -y --no-install-recommends curl ca-certificates

# Download the latest installer
ADD https://astral.sh/uv/install.sh /uv-installer.sh

# Run the installer then remove it
RUN sh /uv-installer.sh && rm /uv-installer.sh

# Ensure the installed binary is on the `PATH`
ENV PATH="/root/.local/bin/:$PATH"

# Copy the relevant files into the image
COPY src/ src/
COPY pyproject.toml pyproject.toml

# Build argument to control which dependency group to install
ARG EXTRA_DEP_GROUP

# Sync the project into a new environment
RUN uv sync --no-dev --extra ${EXTRA_DEP_GROUP}

# Empty bash command
CMD []
```

`build.sh` (could be placed in a Makefile)
```bash
docker build \
  --build-arg EXTRA_DEP_GROUP=feature_engineering \
  -t my_ml_project_feature_engineering:latest .
docker push my_container_reigstry.azurecr.io/my_ml_project_feature_engineering:latest

docker build \
  --build-arg EXTRA_DEP_GROUP=training  \
  -t my_ml_project_training:latest .
docker push my_container_reigstry.azurecr.io/my_ml_project_training:latest

docker build \
  --build-arg EXTRA_DEP_GROUP=inference \
  -t my_ml_project_featureinference .
docker push my_container_reigstry.azurecr.io/my_ml_project_inference:latest

```

`environment.py`

```python
from azure.ai.ml.entities import Environment

feature_engineering_env = Environment(
    image="my_container_reigstry.azurecr.io/my_ml_project_feature_engineering:latest",
    name="my_feature_engineering_environment",
    description="Environment created from a custom Docker image for feature engineering",
)
ml_client.environments.create_or_update(feature_engineering_env)

training_env = Environment(
    image="my_container_reigstry.azurecr.io/my_ml_project_training:latest",
    name="my_training_environment",
    description="Environment created from a custom Docker image for training",
)
ml_client.environments.create_or_update(training_env)

inference_env = Environment(
    image="my_container_reigstry.azurecr.io/my_ml_project_inference:latest",
    name="my_inference_environment",
    description="Environment created from a custom Docker image for inference",
)
ml_client.environments.create_or_update(inference_env)
```

### **Command Jobs**
Command jobs are containerised steps which can be used for various functions within the ML process including:
- Data ingestion
- Data processing / feature engineering
- Model training

To execute a command job you need to set up two scripts:
1. The python script which actually performs the ingestion/processing/training etc.
2. The command job script which tells AML how to run the script (i.e. environment, compute instance etc.)

You run the command job and it will invoke the underlying script. 

The example below shows how you could ingest data into AML from a datastore corresponding to a Fabric lakehouse and register it as a versioned data asset. This approach could be adapted to run data processing or model training steps. 

`ingest.py`

```python
import argparse
import os
import shutil
from pathlib import Path

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", required=True)
    parser.add_argument("--output", required=True)
    args = parser.parse_args()

    source = Path(args.input)
    destination = Path(args.output)
    destination.mkdir(parents=True, exist_ok=True)

    if source.is_file():
        shutil.copy2(source, destination / source.name)
    else:
        for item in source.iterdir():
            target = destination / item.name
            if item.is_dir():
                shutil.copytree(item, target, dirs_exist_ok=True)
            else:
                shutil.copy2(item, target)

if __name__ == "__main__":
    main()


`ingest_command_job.py`

```python
from azure.ai.ml import MLClient, command, Input, Output
from azure.ai.ml.constants import AssetTypes, InputOutputModes
from azure.identity import DefaultAzureCredential
from datetime import datetime

SUBSCRIPTION_ID = "<sub-id>"
RESOURCE_GROUP = "<rg>"
WORKSPACE_NAME = "<aml-workspace>"

ml_client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION_ID,
    RESOURCE_GROUP,
    WORKSPACE_NAME,
)

# 1) Source in Fabric lakehouse through your registered datastore
fabric_input_path = (
    "azureml://datastores/fabric_lakehouse_ds/paths/<lakehouse-subfolder-or-file>"
)

# 2) Snapshot target in workspace default blob store
snapshot_output_path = (
    "azureml://datastores/workspaceblobstore/paths/snapshots/fabric_snapshot_v1/"
)

# 3) Data asset metadata
SNAPSHOT_ASSET_NAME = "fabric_raw_snapshot"
SNAPSHOT_ASSET_VERSION = datetime.utcnow().strftime("%Y%m%d%H%M%S")  # or could use a number/name

copy_job = command(
    code="./src",
    command="python copy_data.py --input ${{inputs.raw_data}} --output ${{outputs.snapshot}}",
    inputs={
        "raw_data": Input(
            type=AssetTypes.URI_FOLDER,   # use URI_FILE if the source is a single file
            path=fabric_input_path,
            mode=InputOutputModes.RO_MOUNT,   # or DOWNLOAD
        )
    },
    outputs={
        "snapshot": Output(
            type=AssetTypes.URI_FOLDER,
            path=snapshot_output_path,
            mode=InputOutputModes.UPLOAD,
            name=SNAPSHOT_ASSET_NAME,         # this registers the output as a data asset
            version=SNAPSHOT_ASSET_VERSION, 
            description="Raw snapshot copied from Microsoft Fabric lakehouse",
        )
    },
    environment="my_environment",
    compute="my_compute",
    display_name="copy-fabric-snapshot",
)

returned_job = ml_client.jobs.create_or_update(copy_job)

print(f"Copy job name: {returned_job.name}")
print(f"Registered asset name: {returned_job.outputs['snapshot'].name}")
print(f"Registered asset version: {returned_job.outputs['snapshot'].version}")
```

### **Hyperparameter Tuning**
If you are just training one model you can just run it as a single command job. However, often you will want to run hyperparameter tuning to train many models; this can be acheived using a sweep job. These work similarly to command job in that you create a script which runs the code and a script to tell AML how to run the code. It is vital when running a sweep job to register the parameters and the metric used to decide 'best' with MLFlow. The sweep job is then configured to point to this registered metric. 

An example of a simple classifier run as a sweep job is shown below. 

`train.py`

```python
import argparse
from pathlib import Path

import joblib
import mlflow
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--training_data", type=str, required=True)
    parser.add_argument("--n_estimators", type=int, default=100)
    parser.add_argument("--max_depth", type=int, default=10)
    parser.add_argument("--model_output", type=str, required=True)
    args = parser.parse_args()

    data_path = Path(args.training_data)

    # Example: expect one parquet file in the asset folder
    df = pd.read_parquet(data_path / "features.parquet")

    X = df.drop(columns=["target"])
    y = df["target"]

    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    mlflow.log_param("n_estimators", args.n_estimators)
    mlflow.log_param("max_depth", args.max_depth)

    model = RandomForestClassifier(
        n_estimators=args.n_estimators,
        max_depth=args.max_depth,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)

    preds = model.predict(X_valid)
    acc = accuracy_score(y_valid, preds)

    mlflow.log_metric("validation_accuracy", acc)

    output_dir = Path(args.model_output)
    output_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, output_dir / "model.pkl")

    print(f"validation_accuracy={acc:.6f}")


if __name__ == "__main__":
    main()
```

`train_sweep_job.py`

```python
from azure.ai.ml import MLClient, command, Input, Output
from azure.ai.ml.constants import AssetTypes, InputOutputModes
from azure.ai.ml.sweep import Choice, Objective, Uniform, RandomSamplingAlgorithm
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "<sub-id>"
RESOURCE_GROUP = "<rg>"
WORKSPACE_NAME = "<aml-workspace>"

TRAINING_DATA_ASSET_NAME = "fabric_feature_set"
TRAINING_DATA_ASSET_VERSION = "20260309123000"

ml_client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION_ID,
    RESOURCE_GROUP,
    WORKSPACE_NAME,
)

trial_job = command(
    code="./src",
    command=(
        "python train.py "
        "--training_data ${{inputs.training_data}} "
        "--n_estimators ${{inputs.n_estimators}} "
        "--max_depth ${{inputs.max_depth}} "
        "--model_output ${{outputs.model_output}}"
    ),
    inputs={
        "training_data": Input(
            type=AssetTypes.URI_FOLDER,
            path=f"azureml:{TRAINING_DATA_ASSET_NAME}:{TRAINING_DATA_ASSET_VERSION}",
            mode=InputOutputModes.RO_MOUNT,
        ),
        "n_estimators": Choice(values=[50, 100, 200, 300]),
        "max_depth": Uniform(min_value=4, max_value=20),
    },
    outputs={
        "model_output": Output(
            type=AssetTypes.URI_FOLDER,
            path="azureml://datastores/workspaceblobstore/paths/sweeps/rf_models/",
            mode=InputOutputModes.UPLOAD,
        )
    },
    environment="my_environment",
    compute="my_compute",
)

sweep_job = trial_job.sweep(
    sampling_algorithm=RandomSamplingAlgorithm(seed=42),
    primary_metric="validation_accuracy",
    goal="maximize",
)

sweep_job.set_limits(
    max_total_trials=12,
    max_concurrent_trials=3,
    timeout=7200,
)

sweep_job.display_name = "rf-hyperparameter-sweep"
sweep_job.experiment_name = "fabric-training-sweep"

returned_sweep = ml_client.jobs.create_or_update(sweep_job)
print(f"Submitted sweep job: {returned_sweep.name}")
```

### **Model Registration**
You could reigster a model directly from a hyperparameter tuning job but in a production workflow it is common to rerun a training job separately using the best hyperparameters. Doing so creates a clean training job free from any data subsetting and early stopping logic. The model can then be registerd as a new version if it performs better than the incumbent best model on a held out test set. 

An example of how this can be done is below:

`evlauate_model.py`

```python
import argparse
import json
from pathlib import Path

import joblib
import mlflow
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


def load_model(model_dir: Path):
    model_path = model_dir / "model.pkl"
    if not model_path.exists():
        raise FileNotFoundError(f"Expected model file not found: {model_path}")
    return joblib.load(model_path)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--incumbent_model", type=str, required=True)
    parser.add_argument("--challenger_model", type=str, required=True)
    parser.add_argument("--test_data", type=str, required=True)
    parser.add_argument("--report_output", type=str, required=True)
    parser.add_argument("--target_column", type=str, default="target")
    parser.add_argument("--min_improvement", type=float, default=0.0)
    args = parser.parse_args()

    incumbent_model_dir = Path(args.incumbent_model)
    challenger_model_dir = Path(args.challenger_model)
    test_data_dir = Path(args.test_data)
    report_output_dir = Path(args.report_output)

    report_output_dir.mkdir(parents=True, exist_ok=True)

    data_file = test_data_dir / "features.parquet"
    if not data_file.exists():
        raise FileNotFoundError(f"Expected test data file not found: {data_file}")

    df = pd.read_parquet(data_file)

    if args.target_column not in df.columns:
        raise ValueError(f"Target column '{args.target_column}' not found in test data")

    X_test = df.drop(columns=[args.target_column])
    y_test = df[args.target_column]

    incumbent_model = load_model(incumbent_model_dir)
    challenger_model = load_model(challenger_model_dir)

    incumbent_preds = incumbent_model.predict(X_test)
    challenger_preds = challenger_model.predict(X_test)

    incumbent_metrics = {
        "accuracy": accuracy_score(y_test, incumbent_preds),
        "precision": precision_score(y_test, incumbent_preds, average="weighted", zero_division=0),
        "recall": recall_score(y_test, incumbent_preds, average="weighted", zero_division=0),
        "f1": f1_score(y_test, incumbent_preds, average="weighted", zero_division=0),
    }

    challenger_metrics = {
        "accuracy": accuracy_score(y_test, challenger_preds),
        "precision": precision_score(y_test, challenger_preds, average="weighted", zero_division=0),
        "recall": recall_score(y_test, challenger_preds, average="weighted", zero_division=0),
        "f1": f1_score(y_test, challenger_preds, average="weighted", zero_division=0),
    }

    accuracy_delta = challenger_metrics["accuracy"] - incumbent_metrics["accuracy"]
    promote_challenger = accuracy_delta > args.min_improvement

    mlflow.log_metric("incumbent_accuracy", incumbent_metrics["accuracy"])
    mlflow.log_metric("incumbent_precision", incumbent_metrics["precision"])
    mlflow.log_metric("incumbent_recall", incumbent_metrics["recall"])
    mlflow.log_metric("incumbent_f1", incumbent_metrics["f1"])

    mlflow.log_metric("challenger_accuracy", challenger_metrics["accuracy"])
    mlflow.log_metric("challenger_precision", challenger_metrics["precision"])
    mlflow.log_metric("challenger_recall", challenger_metrics["recall"])
    mlflow.log_metric("challenger_f1", challenger_metrics["f1"])

    mlflow.log_metric("accuracy_delta", accuracy_delta)
    mlflow.log_param("min_improvement", args.min_improvement)
    mlflow.log_param("promote_challenger", str(promote_challenger))

    report = {
        "incumbent_metrics": incumbent_metrics,
        "challenger_metrics": challenger_metrics,
        "accuracy_delta": accuracy_delta,
        "min_improvement": args.min_improvement,
        "promote_challenger": promote_challenger,
        "winner": "challenger" if promote_challenger else "incumbent",
    }

    report_path = report_output_dir / "comparison_report.json"
    with report_path.open("w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    print(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()
```

`evlauate_model_command_job.py`

```python
import json
from datetime import datetime, timezone
from pathlib import Path

from azure.ai.ml import MLClient, command, Input, Output
from azure.ai.ml.constants import AssetTypes, InputOutputModes
from azure.ai.ml.entities import Model
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "<sub-id>"
RESOURCE_GROUP = "<rg>"
WORKSPACE_NAME = "<aml-workspace>"

# Existing registered incumbent model
INCUMBENT_MODEL_NAME = "rf_model"
INCUMBENT_MODEL_VERSION = "3"

# Challenger comes from a completed training job output
CHALLENGER_TRAIN_JOB_NAME = "<training-job-name>"
CHALLENGER_MODEL_OUTPUT_NAME = "model_output"

# Registered test data asset
TEST_DATA_ASSET_NAME = "feature_test_set"
TEST_DATA_ASSET_VERSION = "20260309120000"

# Only register challenger if comparison says it wins
NEW_MODEL_NAME = "rf_model"
NEW_MODEL_VERSION = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")

REPORT_ASSET_NAME = "model_comparison_report"
REPORT_ASSET_VERSION = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")

MIN_IMPROVEMENT = 0.002

ml_client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION_ID,
    RESOURCE_GROUP,
    WORKSPACE_NAME,
)

# Path to challenger model from prior training job output
challenger_model_path = (
    f"azureml://jobs/{CHALLENGER_TRAIN_JOB_NAME}/outputs/"
    f"{CHALLENGER_MODEL_OUTPUT_NAME}/paths/"
)

comparison_job = command(
    code="./src",
    command=(
        "python evaluate_models.py "
        "--incumbent_model ${{inputs.incumbent_model}} "
        "--challenger_model ${{inputs.challenger_model}} "
        "--test_data ${{inputs.test_data}} "
        "--report_output ${{outputs.report_output}} "
        "--target_column target "
        f"--min_improvement {MIN_IMPROVEMENT}"
    ),
    inputs={
        "incumbent_model": Input(
            type=AssetTypes.CUSTOM_MODEL,
            path=f"azureml:{INCUMBENT_MODEL_NAME}:{INCUMBENT_MODEL_VERSION}",
            mode=InputOutputModes.RO_MOUNT,
        ),
        "challenger_model": Input(
            type=AssetTypes.CUSTOM_MODEL,
            path=challenger_model_path,
            mode=InputOutputModes.RO_MOUNT,
        ),
        "test_data": Input(
            type=AssetTypes.URI_FOLDER,
            path=f"azureml:{TEST_DATA_ASSET_NAME}:{TEST_DATA_ASSET_VERSION}",
            mode=InputOutputModes.RO_MOUNT,
        ),
    },
    outputs={
        "report_output": Output(
            type=AssetTypes.URI_FOLDER,
            path=(
                "azureml://datastores/workspaceblobstore/paths/model-comparisons/"
                f"{REPORT_ASSET_NAME}/{REPORT_ASSET_VERSION}/"
            ),
            mode=InputOutputModes.UPLOAD,
            name=REPORT_ASSET_NAME,
            version=REPORT_ASSET_VERSION,
            description="Incumbent vs challenger comparison report",
        )
    },
    environment="my_environment",
    compute="my_compute",
    display_name="compare-incumbent-vs-challenger",
    experiment_name="model-comparison",
)

submitted_job = ml_client.jobs.create_or_update(comparison_job)
print(f"Submitted comparison job: {submitted_job.name}")

# Wait for completion and show logs
ml_client.jobs.stream(submitted_job.name)

# Download the comparison report
download_dir = Path("./downloaded_comparison_report")
download_dir.mkdir(parents=True, exist_ok=True)

ml_client.jobs.download(
    name=submitted_job.name,
    output_name="report_output",
    download_path=str(download_dir),
)

report_path = download_dir / "named-outputs" / "report_output" / "comparison_report.json"
if not report_path.exists():
    raise FileNotFoundError(f"Could not find comparison report at {report_path}")

with report_path.open("r", encoding="utf-8") as f:
    report = json.load(f)

print(json.dumps(report, indent=2))

if report.get("promote_challenger", False):
    model = Model(
        name=NEW_MODEL_NAME,
        version=NEW_MODEL_VERSION,
        path=challenger_model_path,
        type="custom_model",
        description="Promoted challenger model after beating incumbent",
        tags={
            "source_training_job": CHALLENGER_TRAIN_JOB_NAME,
            "incumbent_model_name": INCUMBENT_MODEL_NAME,
            "incumbent_model_version": INCUMBENT_MODEL_VERSION,
            "test_data_asset": TEST_DATA_ASSET_NAME,
            "test_data_version": TEST_DATA_ASSET_VERSION,
            "comparison_job": submitted_job.name,
            "winner": "challenger",
        },
    )

    registered_model = ml_client.models.create_or_update(model)
    print(
        f"Challenger won. Registered model: "
        f"{registered_model.name}:{registered_model.version}"
    )
else:
    print("Challenger did not beat incumbent. No new model version registered.")
```

### **Model Deployment**
AML offers endpoints for both real-time and batch inference. The example below shows how to deploy a batch endpoint which requires:
- `batch_score.py`: A script to run the inference
- `deploy_endpoint.py`: A script for deploying the endpoint in Azure
- `invoke_endpoint.py` A script for running batch inference via the endpoint



`batch_score.py`

```python
import argparse
from pathlib import Path

import joblib
import pandas as pd


def init_model(model_dir: Path):
    model_path = model_dir / "model.pkl"
    if not model_path.exists():
        raise FileNotFoundError(f"Model not found at {model_path}")
    return joblib.load(model_path)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", type=str, required=True)
    parser.add_argument("--input_data", type=str, required=True)
    parser.add_argument("--output_data", type=str, required=True)
    parser.add_argument("--target_column", type=str, default="target")
    args = parser.parse_args()

    model = init_model(Path(args.model))
    input_dir = Path(args.input_data)
    output_dir = Path(args.output_data)
    output_dir.mkdir(parents=True, exist_ok=True)

    data_file = input_dir / "features.parquet"
    if not data_file.exists():
        raise FileNotFoundError(f"Expected input file not found: {data_file}")

    df = pd.read_parquet(data_file)

    # Drop target if present
    X = df.drop(columns=[args.target_column], errors="ignore")

    preds = model.predict(X)

    result = df.copy()
    result["prediction"] = preds
    result.to_parquet(output_dir / "predictions.parquet", index=False)


if __name__ == "__main__":
    main()
```

`deploy_batch_endpoint.py`

```python
from azure.ai.ml import MLClient, Input
from azure.ai.ml.entities import BatchEndpoint, ModelBatchDeployment, BatchRetrySettings
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "<sub-id>"
RESOURCE_GROUP = "<rg>"
WORKSPACE_NAME = "<aml-workspace>"

MODEL_NAME = "rf_model"
MODEL_VERSION = "5"

ENDPOINT_NAME = "rf-batch-endpoint"
DEPLOYMENT_NAME = "blue"

COMPUTE_NAME = "cpu-cluster"
ENVIRONMENT = "my_environment"

ml_client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION_ID,
    RESOURCE_GROUP,
    WORKSPACE_NAME,
)

# Create endpoint
endpoint = BatchEndpoint(
    name=ENDPOINT_NAME,
    description="Batch endpoint for Random Forest inference",
)

ml_client.batch_endpoints.begin_create_or_update(endpoint).result()

# Create deployment
deployment = ModelBatchDeployment(
    name=DEPLOYMENT_NAME,
    description="Batch deployment for registered RF model",
    endpoint_name=ENDPOINT_NAME,
    model=f"azureml:{MODEL_NAME}:{MODEL_VERSION}",
    code_path="./src",
    scoring_script="batch_score.py",
    environment=ENVIRONMENT,
    compute=COMPUTE_NAME,
    instance_count=2,
    max_concurrency_per_instance=2,
    mini_batch_size="10",
    output_action="append_row",
    output_file_name="predictions.csv",
    retry_settings=BatchRetrySettings(max_retries=3, timeout=300),
)

ml_client.batch_deployments.begin_create_or_update(deployment).result()

# Set default deployment
endpoint = ml_client.batch_endpoints.get(ENDPOINT_NAME)
endpoint.defaults.deployment_name = DEPLOYMENT_NAME
ml_client.batch_endpoints.begin_create_or_update(endpoint).result()

print(f"Created batch endpoint: {ENDPOINT_NAME}")
print(f"Created deployment: {DEPLOYMENT_NAME}")
```

`invoke_batch_endpoint.py`

```python
from datetime import datetime, timezone

from azure.ai.ml import MLClient, Input
from azure.ai.ml.constants import AssetTypes, InputOutputModes
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "<sub-id>"
RESOURCE_GROUP = "<rg>"
WORKSPACE_NAME = "<aml-workspace>"

ENDPOINT_NAME = "rf-batch-endpoint"

# Use a versioned data asset as input
INPUT_DATA_ASSET_NAME = "feature_scoring_set"
INPUT_DATA_ASSET_VERSION = "20260309150000"

OUTPUT_PATH = (
    "azureml://datastores/workspaceblobstore/paths/batch-inference/"
    f"{datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')}/"
)

ml_client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION_ID,
    RESOURCE_GROUP,
    WORKSPACE_NAME,
)

job = ml_client.batch_endpoints.invoke(
    endpoint_name=ENDPOINT_NAME,
    inputs={
        "input_data": Input(
            type=AssetTypes.URI_FOLDER,
            path=f"azureml:{INPUT_DATA_ASSET_NAME}:{INPUT_DATA_ASSET_VERSION}",
            mode=InputOutputModes.RO_MOUNT,
        )
    },
    outputs={
        "output_data": OUTPUT_PATH
    },
)

print(f"Batch job name: {job.name}")
print(f"Output path: {OUTPUT_PATH}")

ml_client.jobs.stream(job.name)
```

### **Pipelines**
Pipelines in AML allow you to orchestrate multiple steps together. Each step is registered as a component and then a pipeline job is defined which connects the steps. 

`components.py`

```python
from azure.ai.ml import MLClient, Input, Output, command
from azure.ai.ml.constants import AssetTypes, InputOutputModes
from azure.ai.ml.dsl import pipeline
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "<sub-id>"
RESOURCE_GROUP = "<rg>"
WORKSPACE_NAME = "<workspace>"

ml_client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION_ID,
    RESOURCE_GROUP,
    WORKSPACE_NAME,
)

# 1) Ingest component
ingest_component = command(
    name="ingest_fabric_snapshot",
    display_name="Ingest Fabric snapshot",
    description="Copy data from Fabric/OneLake datastore to workspace blob",
    code="./src",
    command=(
        "python copy_data.py "
        "--input ${{inputs.raw_data}} "
        "--output ${{outputs.snapshot}}"
    ),
    inputs={
        "raw_data": Input(type=AssetTypes.URI_FOLDER)
    },
    outputs={
        "snapshot": Output(type=AssetTypes.URI_FOLDER)
    },
    environment="my_environment",
)

# 2) Process component
process_component = command(
    name="process_raw_snapshot",
    display_name="Process raw snapshot",
    description="Transform raw snapshot into processed feature data",
    code="./src",
    command=(
        "python process_raw_data.py "
        "--input ${{inputs.input_data}} "
        "--output ${{outputs.processed_data}}"
    ),
    inputs={
        "input_data": Input(type=AssetTypes.URI_FOLDER)
    },
    outputs={
        "processed_data": Output(type=AssetTypes.URI_FOLDER)
    },
    environment="my_environment",
)

# 3) Train component
train_component = command(
    name="train_candidate_model",
    display_name="Train candidate model",
    description="Train challenger model from processed feature data",
    code="./src",
    command=(
        "python train.py "
        "--training_data ${{inputs.training_data}} "
        "--model_output ${{outputs.model_output}}"
    ),
    inputs={
        "training_data": Input(type=AssetTypes.URI_FOLDER)
    },
    outputs={
        "model_output": Output(type=AssetTypes.URI_FOLDER)
    },
    environment="my_environment",
)

# 4) Evaluate component
evaluate_component = command(
    name="evaluate_candidate_vs_incumbent",
    display_name="Evaluate challenger vs incumbent",
    description="Compare candidate model with incumbent on a fixed test set",
    code="./src",
    command=(
        "python evaluate_models.py "
        "--incumbent_model ${{inputs.incumbent_model}} "
        "--challenger_model ${{inputs.challenger_model}} "
        "--test_data ${{inputs.test_data}} "
        "--report_output ${{outputs.report_output}} "
        "--target_column target "
        "--min_improvement ${{inputs.min_improvement}}"
    ),
    inputs={
        "incumbent_model": Input(type=AssetTypes.CUSTOM_MODEL),
        "challenger_model": Input(type=AssetTypes.CUSTOM_MODEL),
        "test_data": Input(type=AssetTypes.URI_FOLDER),
        "min_improvement": Input(type="number"),
    },
    outputs={
        "report_output": Output(type=AssetTypes.URI_FOLDER)
    },
    environment="my_environment",
)
```

`pipeline.py`

```python
@pipeline(
    name="fabric_train_eval_pipeline",
    display_name="Fabric -> process -> train -> evaluate",
    description="Ingest from Fabric, process data, train candidate, and compare against incumbent",
)
def training_pipeline(
    fabric_input_path: str,
    incumbent_model_uri: str,
    test_data_uri: str,
    min_improvement: float = 0.002,
):
    ingest_step = ingest_component(
        raw_data=Input(
            type=AssetTypes.URI_FOLDER,
            path=fabric_input_path,
            mode=InputOutputModes.RO_MOUNT,
        )
    )

    process_step = process_component(
        input_data=ingest_step.outputs.snapshot
    )

    train_step = train_component(
        training_data=process_step.outputs.processed_data
    )

    evaluate_step = evaluate_component(
        incumbent_model=Input(
            type=AssetTypes.CUSTOM_MODEL,
            path=incumbent_model_uri,
            mode=InputOutputModes.RO_MOUNT,
        ),
        challenger_model=train_step.outputs.model_output,
        test_data=Input(
            type=AssetTypes.URI_FOLDER,
            path=test_data_uri,
            mode=InputOutputModes.RO_MOUNT,
        ),
        min_improvement=min_improvement,
    )

    return {
        "snapshot": ingest_step.outputs.snapshot,
        "processed_data": process_step.outputs.processed_data,
        "candidate_model": train_step.outputs.model_output,
        "evaluation_report": evaluate_step.outputs.report_output,
    }
```

`submit_pipeline.py`

```python
pipeline_job = training_pipeline(
    fabric_input_path=(
        "azureml://datastores/fabric_lakehouse_ds/paths/<lakehouse-subfolder>"
    ),
    incumbent_model_uri="azureml:rf_model:3",
    test_data_uri="azureml:feature_test_set:20260309120000",
    min_improvement=0.002,
)

pipeline_job.settings.default_compute = "my_compute"

returned_job = ml_client.jobs.create_or_update(
    pipeline_job,
    experiment_name="fabric-train-eval-pipeline",
)

print(f"Submitted pipeline job: {returned_job.name}")
```

### **Finalised Workflow**
Suppose we have data living in a Fabric lakehouse that we would like to use to train a model. The steps to complete this are as follows:
1. Create and register one or more AML environments to use for running the process
2. Register the Fabric lakehouse as an AML datastore
3. Create an ingestion script which queries the data from the datastore and saves a snapshot as a versioned data asset to the default workspace datastore. Run this script as a command job.
4. Create a processing script which take the versionned data asset corresponding to the raw data as input, processes it and conducts feature engineer before saving this as a different versionned data asset. Run this as a command job. 
5. Create a training script which takes the versioned data asset corresponding to the features as input and trains models using hyperparameter tuning. 
6. Define the scripts as components and create a pipeline

Data lives in Fabric where gold is modelled data for BI/analytics purposes and bronze/silver is data to be used for ML
AML has datastore which is a saved connection to the Fabric lakehouse; this is just a connection and doesn't store any data physically
Processing script runs which loads data from datastore (i.e. fabric lakehouse), saves the snapshot to workspace blob storage, processes it to build a features dataset which is also saved to the workplace blob storage
Data assets registered which point to the raw snapshot and features datasets in workspace blob storage
Training job points to data asset for features (typically multiple models trained via hyperparameter tuning). Store the following via MLFlow for reproducibility:
- git SHA for feature and training code
- raw data and features data assets name and version
- environment name and version
Output model artifact from best training job registered in model registry if it performs better than the incumbent model (if one exists)
Deploy the model to an appropriate endpoint (batch / real-time) being careful to think about whether the feature engineering code lives inside the scoring container or upstream from that